Imports

In [1]:
import os
import re
import json
from pathlib import Path
from typing import List, Dict, Any

import fitz  # PyMuPDF
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from dotenv import load_dotenv

Config

In [62]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env")

client = OpenAI(api_key=OPENAI_API_KEY)

PDF_PATH = Path("../data/TPQuebec_Politique_RH.pdf")
CHROMA_PATH = "../chroma_db"
COLLECTION_NAME = "hr_policies"

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

TOP_K = 4
SIMILARITY_THRESHOLD = 0.5

Extraction du PDF

In [5]:
def extract_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    doc = fitz.open(pdf_path)
    pages = []

    for i, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({
            "page_number": i + 1,
            "text": text.strip()
        })

    doc.close()
    return pages

def clean_text(text: str) -> str:
    # Normaliser espaces
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    
    # Réduire les sauts de ligne excessifs
    text = re.sub(r"\n{3,}", "\n\n", text)
    
    # Trim
    return text.strip()

In [6]:
pages = extract_pdf_pages(PDF_PATH)

for page in pages:
    page["text"] = clean_text(page["text"])

print(pages[2]["text"][:2000])  # page 3-ish

3 
Politique RH - Version 1 
 
1- 
Énoncé de politique 
La présente politique des ressources humaines (la « politique RH ») a pour objet de définir un 
mode de gestion des ressources humaines permettant l’accomplissement de la mission, des 
engagements et des orientations de TPQuébec ainsi que le développement et la valorisation des 
personnes qui y travaillent. Les employés de TPQuébec doivent se conformer en tout temps à 
cette politique. 
 
2- 
Dispositions générales 
La publication et la diffusion de la politique RH ainsi que de toute modification qui y sera 
apportée sont confiées à la Direction générale. Tous les employés de TPQuébec doivent y avoir 
accès. La politique RH doit être disponible et accessible en tout temps sur l’intranet ou sur 
demande à la Direction générale. La Direction générale s’assure de l’application, la diffusion et 
la mise à jour de la Politique. 
 
3- Statut de l’employé et période de probation 
3.1 
Statut de l’employé 
Employé temporaire : Un employé 

In [31]:
MAIN_SECTION_PATTERN = re.compile(r"^\s*(\d+)-\s+(.+)$", re.MULTILINE)
SUBSECTION_PATTERN = re.compile(r"^\s*(\d+\.\d+)\s+(.+)$", re.MULTILINE)
PAGE_MARKER_PATTERN = re.compile(r"\[\[PAGE_(\d+)\]\]")

def clean_page_text(text: str) -> str:
    if not text: return ""
    text = re.sub(r"Politique RH\s*-\s*Version\s*1", "", text, flags=re.IGNORECASE) # Retirer le footer/header répété
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE) # Retirer les numéros de page
    text = text.replace("\xa0", " ") # Normaliser les espaces
    text = re.sub(r"[ \t]+", " ", text) # Normaliser les espaces
    text = re.sub(r"\n{3,}", "\n\n", text) # Réduire les lignes vides excessives
    return text.strip()

def build_full_text_with_page_markers(pages: List[Dict[str, Any]]) -> str:
    parts = []
    for page in pages:
        if page["page_number"] == 2:
            continue

        cleaned = clean_page_text(page["text"])
        if not cleaned:
            continue

        parts.append(f"\n\n[[PAGE_{page['page_number']}]]\n{cleaned}")
    return "\n".join(parts).strip()

def find_page_for_index(text: str, char_index: int) -> int:
    matches = list(PAGE_MARKER_PATTERN.finditer(text))
    current_page = 1

    for match in matches:
        if match.start() <= char_index:
            current_page = int(match.group(1))
        else:
            break

    return current_page

def extract_section_chunks(full_text: str) -> List[Dict[str, Any]]:
    main_matches = list(MAIN_SECTION_PATTERN.finditer(full_text))
    chunks = []

    for i, main_match in enumerate(main_matches):
        main_num = main_match.group(1)
        main_title = main_match.group(2).strip()

        start = main_match.start()
        end = main_matches[i + 1].start() if i + 1 < len(main_matches) else len(full_text)

        main_block = full_text[start:end]
        subsection_matches = list(SUBSECTION_PATTERN.finditer(main_block))

        # Si on a des sous-sections, chunk par sous-section
        if subsection_matches:
            for j, sub_match in enumerate(subsection_matches):
                sub_num = sub_match.group(1)
                sub_title = sub_match.group(2).strip()

                sub_start = start + sub_match.start()
                sub_end = (
                    start + subsection_matches[j + 1].start()
                    if j + 1 < len(subsection_matches)
                    else end
                )

                chunk_text = strip_page_markers(full_text[sub_start:sub_end])
                chunks.append(create_chunk_dict(chunk_text, sub_num, sub_title, main_num, full_text, sub_start))
        else:
            # Pas de sous-section -> garder la section principale
            chunk_text = strip_page_markers(main_block)
            if is_meaningful_chunk(chunk_text):
                chunks.append(create_chunk_dict(chunk_text, main_num, main_title, None, full_text, start))
    return chunks

def create_chunk_dict(text, num, title, parent, full_text, char_idx):
    page_number = find_page_for_index(full_text, char_idx)
    return {
        "id": f"section_{num.replace('.', '_')}_p{page_number}",
        "section_number": num,
        "section_title": title,
        "parent_section": parent,
        "page_number": page_number,
        "text": text
    }

def strip_page_markers(text: str) -> str:
    text = PAGE_MARKER_PATTERN.sub("", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def is_meaningful_chunk(text: str) -> bool:
    if not text: return False
    normalized = text.strip()
    if len(normalized) < 80: return False # Trop court

    # Ligne type table des matières (au cas où)
    if re.search(r"\.{5,}\s*\d+\s*$", normalized):
        return False

    return True

def split_large_chunk(chunk, max_chars=1200, overlap=150):
    text = chunk["text"]
    
    # If it fits, don't touch it. (Saves processing & storage)
    if len(text) <= max_chars:
        return [chunk]

    parts = []
    start = 0
    part_idx = 1
    
    while start < len(text):
        end = start + max_chars
        
        # If we aren't at the very end, try to snap to the nearest space
        if end < len(text):
            # Look for a space in the last 100 chars of this window
            last_space = text.rfind(" ", end - 100, end)
            if last_space != -1:
                end = last_space

        sub_text = text[start:end].strip()
        
        # Create the sub-chunk
        new_part = chunk.copy()
        new_part["id"] = f"{chunk['id']}_pt{part_idx}"
        new_part["text"] = sub_text
        parts.append(new_part)
        
        # Move forward: Next start is 'end minus overlap'
        start = end - overlap
        part_idx += 1
        
        # Safety break: if the remaining text is tiny, just stop
        if len(text) - start < 50:
            break
            
    return parts

In [ ]:
full_text = build_full_text_with_page_markers(pages)
initial_chunks = extract_section_chunks(full_text)
print(f"Initially: {len(initial_chunks)} chunks per section.")
final_chunks = []
for chunk in initial_chunks:
    split_parts = split_large_chunk(chunk, max_chars=1200, overlap=100)
    final_chunks.extend(split_parts)

print(f"Generated {len(final_chunks)} chunks to keep chunks a low size.")

Initially: 24 chunks for the Vector DB.
Generated 31 chunks for the Vector DB.


In [ ]:
print(f"Total useful chunks: {len(final_chunks)}\n")

for chunk in final_chunks:
    print("ID:", chunk["id"])
    print("Section:", chunk["section_number"], "-", chunk["section_title"])
    print("Page:", chunk["page_number"])
    print(chunk["text"][:500], "\n")

Base de données de vecteur - Chroma

In [ ]:
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=EMBEDDING_MODEL
)

In [36]:
try:
    chroma_client.delete_collection("hr_policies_tpquebec")
except Exception:
    print("No existing collection to delete")


if COLLECTION_NAME in [c.name for c in chroma_client.list_collections()]:
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
else:
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        embedding_function=openai_ef
    )

In [37]:
documents = []
metadatas = []
ids = []

for chunk in final_chunks:
    documents.append(chunk["text"])
    ids.append(chunk["id"])
    metadatas.append({
        "section_number": chunk["section_number"],
        "section_title": chunk["section_title"],
        "parent_section": chunk["parent_section"] if chunk["parent_section"] else "",
        "page_number": chunk["page_number"],
        "source": PDF_PATH.name
    })

print("Documents:", len(documents))
print("Metadata:", len(metadatas))
print("IDs:", len(ids))

Documents: 31
Metadata: 31
IDs: 31


In [38]:
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print("Chunks indexed successfully!")
print("Collection count:", collection.count())

Chunks indexed successfully!
Collection count: 31


In [42]:
def retrieve_chunks(question: str, top_k: int = TOP_K) -> Dict[str, Any]:
    results = collection.query(
        query_texts=[question],
        n_results=top_k
    )
    return results

def format_retrieved_context(results: Dict[str, Any]) -> List[Dict[str, Any]]:
    formatted = []

    ids = results["ids"][0]
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]

    for doc_id, doc, meta, distance in zip(ids, docs, metas, distances):
        formatted.append({
            "id": doc_id,
            "text": doc,
            "section_number": meta["section_number"],
            "section_title": meta["section_title"],
            "page_number": meta["page_number"],
            "source": meta["source"],
            "distance": distance
        })

    return formatted

In [ ]:
test_question = "Quelle est la période de probation pour un employé cadre ?"

results = retrieve_chunks(test_question, top_k=4)

for i in range(len(results["ids"][0])):
    print("ID:", results["ids"][0][i])
    print("Distance:", results["distances"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print("Excerpt:", results["documents"][0][i][:500], '\n')

ID: section_3_2_p3
Distance: 0.25048160552978516
Metadata: {'page_number': 3, 'section_title': 'Période de probation', 'parent_section': '3', 'section_number': '3.2', 'source': 'TPQuebec_Politique_RH.pdf'}
Excerpt: 3.2 
Période de probation 
Dès son embauche, tout employé est soumis à une période de probation de 3 mois de service 
continu. Pour les postes de niveau supérieur (cadre), la période de probation prévue est de 6 
mois. 
Une rencontre d’échanges est effectuée entre la Direction générale et l’employé au terme de 
cette période. Dans le cas où certains ajustements doivent être apportés, TPQuébec peut 
prolonger cette période de probation d’un autre trois mois.

ID: section_12_2_p9
Distance: 0.49260085821151733
Metadata: {'page_number': 9, 'section_number': '12.2', 'section_title': 'Vérification des antécédents judiciaires', 'parent_section': '12', 'source': 'TPQuebec_Politique_RH.pdf'}
Excerpt: 12.2 
Vérification des antécédents judiciaires 
 
TPQuébec procédera à une vérificat

In [43]:
def should_refuse_answer(retrieved_chunks: List[Dict[str, Any]], threshold: float = SIMILARITY_THRESHOLD) -> bool:
    if not retrieved_chunks:
        return True

    best_distance = retrieved_chunks[0]["distance"]
    return best_distance > threshold

In [44]:
def build_context_for_llm(retrieved_chunks: List[Dict[str, Any]]) -> str:
    context_parts = []

    for idx, chunk in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"""[SOURCE {idx}]
                Document: {chunk['source']}
                Section: {chunk['section_number']} - {chunk['section_title']}
                Page: {chunk['page_number']}

                {chunk['text']}
            """
        )

    return "\n\n".join(context_parts)

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))
from backend.config import SYSTEM_PROMPT

def generate_grounded_answer(question: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    context = build_context_for_llm(retrieved_chunks)

    user_prompt = f"""
                    Question:
                    {question}

                    Context:
                    {context}
                   """

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
def ask_hr_policy(question: str, top_k: int = TOP_K, threshold: float = SIMILARITY_THRESHOLD) -> Dict[str, Any]:
    raw_results = retrieve_chunks(question, top_k=top_k)
    retrieved_chunks = format_retrieved_context(raw_results)

    if should_refuse_answer(retrieved_chunks, threshold=threshold):
        return {
            "question": question,
            "answer": "Je n'ai pas trouvé cette information dans le document de politique RH actuel.",
            "sources": [],
            "retrieved_chunks": retrieved_chunks
        }

    answer = generate_grounded_answer(question, retrieved_chunks)

    sources = [
        {
            "section_number": chunk["section_number"],
            "section_title": chunk["section_title"],
            "page_number": chunk["page_number"]
        }
        for chunk in retrieved_chunks
    ]

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": retrieved_chunks
    }

Test 1 (Question qui devrait être répondue avec un bon score)

In [ ]:
response = ask_hr_policy("Quelle est la période de probation pour un employé cadre ?")

print("QUESTION:\n", response["question"])
print("\nANSWER:\n", response["answer"])
print("\nRETRIEVED SOURCES:")
for s in response["sources"]:
    print(f"- Section {s['section_number']} - {s['section_title']} (Page {s['page_number']})")

print("\nBest distances:")
for chunk in response["retrieved_chunks"]:
    print(chunk["distance"], "|", chunk["section_number"], "-", chunk["section_title"])

QUESTION:
 Quelle est la période de probation pour un employé cadre ?

ANSWER:
 Answer:
La période de probation pour un employé cadre est de 6 mois de service continu (Section 3.2).

Sources:
- Section 3.2 (Page 3)

RETRIEVED SOURCES:
- Section 3.2 - Période de probation (Page 3)
- Section 12.2 - Vérification des antécédents judiciaires (Page 9)
- Section 5.1 - Horaire de travail (Page 5)
- Section 3.1 - Statut de l’employé (Page 3)


In [61]:
print("\nBest distances:")
for chunk in response["retrieved_chunks"]:
    print(chunk["distance"], "|", chunk["section_number"], "-", chunk["section_title"])


Best distances:
0.25048160552978516 | 3.2 - Période de probation
0.49260085821151733 | 12.2 - Vérification des antécédents judiciaires
0.5035502910614014 | 5.1 - Horaire de travail
0.5257024168968201 | 3.1 - Statut de l’employé


Test 2 (autre bonne question)

In [57]:
response = ask_hr_policy("Les employés à salaire annuel ont-ils droit aux heures supplémentaires ?")

print(response["answer"])

Answer:
Les employés à salaire annuel ne sont pas admissibles aux heures supplémentaires, selon la politique énoncée dans la section 4.5.

Sources:
- Section 4.5 (Page 4)


Test 3 (question sur les vacances)

In [58]:
response = ask_hr_policy("Combien de semaines de vacances après 3 ans de service ?")

print(response["answer"])

Answer:
Après trois années de service, le nombre de semaines de vacances est augmenté à trois semaines (Section 5.2).

Sources:
- Section 5.2 (Page 5)


Test 4 (Question qui ne devrait pas être répondue / Pas dans le scope)

In [59]:
response = ask_hr_policy("Quelle est la politique de congé parental ?")

print(response["answer"])
print("\nBest distances:")
for chunk in response["retrieved_chunks"]:
    print(chunk["distance"], "|", chunk["section_number"], "-", chunk["section_title"])

Answer:
Je n'ai pas trouvé d'informations sur la politique de congé parental dans le document de politique RH actuel.

Sources:
- Section 5.2 (Page 5)
- Section 4.5 (Page 4)
- Section 5.1 (Page 5)
- Section 2 (Page 3)

Best distances:
0.4419553279876709 | 5.2 - Congés
0.550363302230835 | 4.5 - Temps supplémentaire et banque d’heures
0.5678561925888062 | 5.1 - Horaire de travail
0.5830365419387817 | 2 - Dispositions générales


Selon les distances observé, le seuil de similarity_threshold devrait être diminué à 0.5 ce qui élimine les sections vraiment pas en rapport